In [ ]:
#| default_exp j

# j
> Running J in-process through libj, and `j` magics for Jupyter and IPython

In [ ]:
import ctypes
from ctypes import c_void_p,c_char_p,c_int,c_longlong,c_double,c_byte,POINTER,byref,cast,create_string_buffer
from shutil import which
from IPython.display import display
from fastcore.utils import *
from fastcore.test import *

jnb runs J in the Python process by loading libj, the engine used by `jconsole`. Direct calls report completion and errors without parsing console prompts. Output arrives through a callback. Another thread can interrupt a computation with `JInterrupt`.

This notebook builds the binding up one step at a time. Install J with `pip install jlanguage` or a Jsoftware distribution. The engine library and `profile.ijs`, which loads the standard library, are in the installation's `bin` directory.

## Finding J

In [ ]:
_LIBJ = 'j.dll' if sys.platform=='win32' else 'libj.dylib' if sys.platform=='darwin' else 'libj.so'

def find_j():
    "Locate the J binary directory: the one containing libj and profile.ijs"
    if (p:=which('jconsole')) and (d:=Path(p).resolve().parent/_LIBJ).exists(): return d.parent
    try:
        import jlang
        return Path(jlang.path())/'bin'
    except ImportError: pass
    roots = [Path.home(), Path('/Applications'), Path('/opt'), Path('/usr/share')]
    cands = sorted(c for r in roots if r.exists() for c in r.glob('j9*') if (c/'bin'/_LIBJ).exists())
    if cands: return cands[-1]/'bin'
    raise FileNotFoundError('J not found: pip install jlanguage, or install it from jsoftware.com')

`find_j` checks for libj beside `jconsole`, resolving symlinks first. The `jlanguage` console script is a Python wrapper without libj beside it. For that installation, `find_j` obtains the path from `jlang`. It also checks common installation directories.

In [ ]:
find_j()

Path('/Users/jhoward/aai-ws/.venv/lib/python3.13/site-packages/jlang/bin')

## The engine

In [ ]:
_OUTCB = ctypes.CFUNCTYPE(None, c_void_p, c_int, c_void_p)   # Joutput(jt, type, text)
_INCB  = ctypes.CFUNCTYPE(c_void_p, c_void_p, c_char_p)      # Jinput(jt, prompt) -> line
_SMCON = 3   # smoptions: identify as a console front end

def load_j(jbin=None):
    "Load libj from `jbin` (or `find_j()`), declare its C signatures, and return `(jbin,lib)`"
    jbin = Path(jbin or find_j())
    lib = ctypes.CDLL(str(jbin/_LIBJ))
    lib.JInit2.restype = c_void_p
    lib.JInit2.argtypes = [c_char_p]
    lib.JDo.argtypes = [c_void_p,c_char_p]
    lib.JSM.argtypes = [c_void_p,c_void_p]
    lib.JInterrupt.argtypes = lib.JFree.argtypes = [c_void_p]
    lib.JGetM.argtypes = lib.JSetM.argtypes = [c_void_p,c_char_p]+[POINTER(c_longlong)]*4
    return jbin,lib

J declares its C API in `jsource/jsrc/jlib.h`. `JInit2` creates an engine with its installation path. The engine needs this path to find `libgmp` for extended precision. Do not substitute `JInit`. Mixing the two initializers in one process can cause a crash.

`JDo` executes one line. `JSM` registers five callback slots: `{output, wd, input, unused, options}`. We use the same arrangement as `jconsole`: `{Joutput, 0, Jinput, 0, SMCON}`.

The output callback receives a type and a pointer. These types need different handling:

- Type 1 contains formatted output.
- Type 2 contains an error display.
- Type 5 requests exit through `2!:55`. The pointer value is the exit code, including NULL for zero.

The input callback supplies continuation lines for multi-line definitions. The first example only collects output in a list.

In [ ]:
jbin,lib = load_j()
jt = lib.JInit2(str(jbin).encode())
out = []
@_OUTCB
def outcb(j,typ,p): out.append((typ, ctypes.string_at(p).decode('utf-8','replace') if p else p))
cbs = (c_void_p*5)(cast(outcb,c_void_p), None, None, None, _SMCON)
lib.JSM(jt, cbs)
lib.JDo(jt, b'+/ % # 1 2 3 4'), out

(0, [(1, '0.25\n')])

`JDo` returns zero on success or a J error number on failure. Output callbacks finish before it returns.

A new engine has no standard library. The following sentence sets `BINPATH` and `ARGV`, then runs `profile.ijs`, following `jefirst` in `jsource/jsrc/jeload.c`. J evaluates right to left, starting with `BINPATH`.

In [ ]:
out.clear()
boot = f"(3 : '0!:0 y')<BINPATH,'/profile.ijs'[ARGV_z_=:<'jconsole'[BINPATH_z_=:'{jbin}'"
lib.JDo(jt, boot.encode()), out

(0, [])

Call `toupper` from the standard library.

In [ ]:
out.clear()
lib.JDo(jt, b"toupper 'j is here'"), out

(0, [(1, 'J IS HERE\n')])

## Errors

A domain error returns error number 3 and a type-2 output containing the explanation. `JError` uses this output as its exception message.

In [ ]:
out.clear()
lib.JDo(jt, b"1 + 'a'"), out

(3, [(2, "|domain error, executing dyad +\n|y is character\n|   1    +'a'\n")])

In [ ]:
class JError(Exception):
    "A J error, carrying the session's output (including the error display) as its message"

## A session object

`J` creates an engine with callbacks and the standard library loaded. It collects output as `(type, text)` pairs and records exit requests separately. The output callback handles NULL pointers.

`run` queues lines of J code. It sends top-level lines through `JDo`. When J opens a multi-line definition, the input callback supplies its body from the same queue.

The input buffer must remain alive after the callback returns. The engine reads it afterward. `J` keeps the buffer on the instance to avoid the memory leak caused by returning Python `bytes` directly from a ctypes callback.

In [ ]:
class J:
    "A J session: the libj engine loaded in-process, with the stdlib profile booted"
    def __init__(self, jbin=None):
        self.jbin,self._lib = load_j(jbin)
        self.jt = self._lib.JInit2(str(self.jbin).encode())
        self._out,self._inq,self.exited = [],[],None
        @_OUTCB
        def _o(j,typ,p):
            if typ==5: self.exited = p or 0
            else: self._out.append((typ, ctypes.string_at(p).decode('utf-8','replace') if p else ''))
        @_INCB
        def _i(j,prompt):
            self._inbuf = create_string_buffer((self._inq.pop(0) if self._inq else ')').encode())
            return ctypes.addressof(self._inbuf)
        self._cbs = (c_void_p*5)(cast(_o,c_void_p), None, cast(_i,c_void_p), None, _SMCON)
        self._lib.JSM(self.jt, self._cbs)
        boot = f"(3 : '0!:0 y')<BINPATH,'/profile.ijs'[ARGV_z_=:<'jconsole'[BINPATH_z_=:'{self.jbin}'"
        if self._lib.JDo(self.jt, boot.encode()): raise JError(''.join(t for _,t in self._out))

`run` returns the session's output. A J error raises `JError` with that output, including the error display.

In [ ]:
@patch
def run(self:J, code):
    "Run `code` (one or more lines of J) in the session, returning its output; raises `JError` on J errors"
    self._out.clear()
    self._inq[:] = code.strip().splitlines()
    while self._inq and self.exited is None:
        rc = self._lib.JDo(self.jt, self._inq.pop(0).encode())
        if rc and self.exited is None: raise JError(''.join(t for _,t in self._out))
    if self.exited is not None: return ''.join(t for ty,t in self._out if ty!=2)   # drop the exit unwind noise
    return ''.join(t for _,t in self._out)

In [ ]:
j = J()
print(j.run('m =: 2 3 $ 10 * 1 + i. 6\nm'))

10 20 30
40 50 60



Check that state persists, multi-line definitions work, and errors raise `JError`.

In [ ]:
test_eq(j.run('+/ , m'), '210\n')
test_eq(j.run('mean =: 3 : 0\n(+/ y) % # y\n)\nmean 1 2 3 4'), '2.5\n')
test_fail(lambda: j.run("m + 'x'"), contains='domain error')

Call the session directly to display output without `print`. The result is a `JOut` string with a verbatim representation. String comparison and slicing still work. A call without output returns `None`.

In [ ]:
class JOut(str):
    "Output text from a `J` call; displays verbatim"
    def __repr__(self): return str(self)

@patch
def __call__(self:J, code):
    "Run `code`, returning session output (or None if there is none)"
    return JOut(self.run(code)) or None

In [ ]:
j('m ,. |. m')

10 20 30 40 50 60
40 50 60 10 20 30

In [ ]:
test_eq(j('+/ , m'), '210\n')
test_is(j('m2 =: 10 * m'), None)

Calculate 25 factorial with extended precision.

In [ ]:
j('*/ 1 + i. 25x')

15511210043330985984000000

## Getting values into Python

Use `getm` to read a named J noun as Python data. `run` returns display text.

`JGetM` provides the noun's type, rank, dimensions and a pointer to its flat data. `getm` converts characters to strings and numeric arrays to nested lists. Booleans, integers and floats are supported. Boxed, extended and rational values raise `JError`.

In [ ]:
def _nest(x, shape):
    "Nest flat sequence `x` (row-major ravel) into `shape`"
    if len(shape)<2: return x
    n = len(x)//shape[0]
    return [_nest(x[i*n:(i+1)*n], shape[1:]) for i in range(shape[0])]

@patch
def getm(self:J, name):
    "Read noun `name` into Python: str for characters; int/float scalars and (nested) lists otherwise"
    t,r,s,d = c_longlong(),c_longlong(),c_longlong(),c_longlong()
    if self._lib.JGetM(self.jt, name.encode(), byref(t),byref(r),byref(s),byref(d)): raise JError(f'no noun: {name}')
    shape = cast(s.value, POINTER(c_longlong))[:r.value]
    n = math.prod(shape)
    if t.value==2: return _nest(ctypes.string_at(d.value, n).decode('utf-8','replace'), shape)
    if t.value not in (1,4,8): raise JError(f'unsupported J type {t.value} for: {name}')
    x = cast(d.value, POINTER({1:c_byte,4:c_longlong,8:c_double}[t.value]))[:n]
    return _nest(x, shape) if r.value else x[0]

`j[expr]` calls `pyval` to evaluate an expression and return Python data. It temporarily assigns the result to `jnbtmp`, reads it, then erases that noun.

In [ ]:
@patch
def pyval(self:J, expr):
    "Evaluate `expr` and return the result as a Python value"
    self.run(f'jnbtmp =: {expr}')
    try: return self.getm('jnbtmp')
    finally: self.run("4!:55 <'jnbtmp'")

@patch
def __getitem__(self:J, expr): return self.pyval(expr)

In [ ]:
test_eq(j['m'], [[10,20,30],[40,50,60]])
test_eq(j['m > 25'], [[0,0,1],[1,1,1]])
test_eq(j['+/ % # 1 2 3 4'], 0.25)
test_eq(j["'py' , 'val'"], 'pyval')

Assign Python data with `j[name] = value`. `JSetM` accepts the typed buffers built by `_jdat`. Strings become J character arrays. Numeric arrays use integers unless any element is a float.

`fn` turns a J verb into a Python callable. Pass one argument for monadic use or two for dyadic use, with the left argument first.

In [ ]:
def _flat(o): return [a for x in o for a in _flat(x)] if isinstance(o,(list,tuple)) else [o]

def _jdat(v):
    "ctypes `(type,shape,buf)` for a Python scalar, string, or uniformly nested list"
    if isinstance(v,str):
        b = v.encode()
        return 2, [len(b)], create_string_buffer(b, len(b))
    shape,x = [],v
    while isinstance(x,(list,tuple)): shape,x = shape+[len(x)],x[0]
    flat = _flat(v)
    if any(isinstance(a,float) for a in flat): return 8, shape, (c_double*len(flat))(*flat)
    return 4, shape, (c_longlong*len(flat))(*flat)

@patch
def __setitem__(self:J, nm, v):
    "Assign Python value `v` (scalar, string, or nested list) to noun `nm`"
    t,shape,buf = _jdat(v)
    sh = (c_longlong*len(shape))(*shape)
    a = [c_longlong(t), c_longlong(len(shape)), c_longlong(ctypes.addressof(sh)), c_longlong(ctypes.addressof(buf))]
    if self._lib.JSetM(self.jt, nm.encode(), *map(byref,a)): raise JError(f'JSetM failed: {nm}')

In [ ]:
j['q'] = [[1,2],[3,4.5]]
test_eq(j['q'], [[1,2],[3,4.5]])
j['s'] = "it's"
test_eq(j['s'], "it's")
test_eq(j['+/ , q'], 10.5)

In [ ]:
@patch
def fn(self:J, code):
    "A Python callable applying J verb `code` monadically or dyadically (left argument first)"
    def f(*args):
        if len(args)==1:
            self['jnby'] = args[0]
            return self.pyval(f'({code}) jnby')
        self['jnbx'],self['jnby'] = args
        return self.pyval(f'jnbx ({code}) jnby')
    return f

In [ ]:
sq = j.fn('*:')
test_eq(sq([1,2,3]), [1,4,9])
test_eq(j.fn('+/')([1,2,3]), 6)
test_eq(j.fn('{.')(2, [5,6,7]), [5,6])

## Interrupting

Call `interrupt` from another thread to stop a running computation. The thread inside `JDo` cannot run Python signal handlers until the C call returns. The jkernel worker handles SIGINT through another thread.

`JInterrupt` sets the engine's break flag. J reports an attention interrupt error, after which the session remains usable.

In [ ]:
@patch
def interrupt(self:J):
    "Stop the currently running sentence with an attention interrupt; safe to call from another thread"
    self._lib.JInterrupt(self.jt)

In [ ]:
import threading,time

In [ ]:
j('spin =: 3 : 0\nn =. 0\nwhile. n < 1e8 do. n =. n + 1 end.\n)')
t0 = time.time()
threading.Timer(0.5, j.interrupt).start()
test_fail(lambda: j('spin 0'), contains='attention interrupt')
assert time.time()-t0 < 5
test_eq(j('2+2'), '4\n')

## Exit requests and shutdown

`exit 0` calls the `2!:55` foreign to request exit. `J` records the requested code in `.exited`. `run` stops processing queued lines and returns without raising. The host, such as jkernel, decides how to handle the request.

The engine runs inside Python, with no child process to leave running. `close` frees its memory. A `with` block calls `close` on exit. Do not use a closed session.

In [ ]:
@patch
def close(self:J):
    "Free the engine instance; the session is unusable afterwards"
    self._lib.JFree(self.jt)
    self.jt = None

@patch
def __enter__(self:J): return self

@patch
def __exit__(self:J, *args): self.close()

In [ ]:
with J() as j2:
    test_is(j2.exited, None)
    j2('exit 7')
    test_eq(j2.exited, 7)

## The `j` magics

Load the extension with `%load_ext jnb`. It registers two magics:

- `%%j` runs a cell and displays its output verbatim. A trailing `;` suppresses the display.
- `%j expr` returns Python data, as `j[expr]` does. You can assign the result, for example `z = %j z`.

The engine starts on the first call to a magic.

In [ ]:
class JMagic:
    "IPython `%j`/`%%j` magics, driving a lazily-started `J` session"
    def __init__(self, jbin=None): self.jbin,self.o = jbin,None

    def j(self, line, cell=None):
        "Run J: a cell magic displays the session output; a line magic returns the expression's Python value"
        if not self.o: self.o = J(self.jbin)
        if cell is None: return self.o[line.strip()]
        disp,cell = True,cell.rstrip()
        if cell.endswith(';'): disp,cell = False,cell[:-1]
        out = self.o(cell)
        if disp and out: display(out)

In [ ]:
def create_j_magic(shell=None):
    "Create a `JMagic` and register its `j` line/cell magic with `shell`, returning it"
    if not shell: shell = get_ipython()
    jm = JMagic()
    shell.register_magic_function(jm.j, 'line_cell', 'j')
    return jm

def load_ipython_extension(ipython):
    "Required function for creating magic"
    create_j_magic(shell=ipython)

In [ ]:
# Only required if you don't load the extension
magic = create_j_magic()

In [ ]:
#| export
def load_ipython_extension(ipython):
    "Register the `j` magics: `%load_ext jnb`"
    create_j_magic(shell=ipython)

def create_ipython_config():
    "Called by `jnb_install` to register the extension in every IPython and Jupyter session"
    from IPython.paths import get_ipython_dir
    cf = Path(get_ipython_dir())/'profile_default'/'ipython_config.py'
    cf.parent.mkdir(parents=True, exist_ok=True)
    if cf.exists() and 'jnb' in cf.read_text(): return print('jnb already installed!')
    with cf.open(mode='a') as f: f.write("\nc.InteractiveShellApp.extensions.append('jnb')\n\n")
    print(f"Jupyter config updated at {cf}")

In [ ]:
%%j
m3 =: 3 3 $ i. 9
m3 +/ . * m3

15 18  21
42 54  66
69 90 111

Assign a J result to a Python variable, then suppress a cell's output.

In [ ]:
z = %j m3
test_eq(z, [[0,1,2],[3,4,5],[6,7,8]])

In [ ]:
%%j
big =: 1000 1000 $ i. 5
big + big;

In [ ]:
#| hide
from IPython.utils.capture import capture_output

In [ ]:
#| hide
with capture_output() as cap: magic.j('', '2+2;')
test_eq(len(cap.outputs), 0)
with capture_output() as cap: magic.j('', '2+2')
test_eq(len(cap.outputs), 1)

## Cleanup

Free the engines created by these examples.

In [ ]:
if magic.o: magic.o.close()
j.close()
lib.JFree(jt)

0

## Export -

In [ ]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()